In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN loaded securely from Kaggle Secrets.")
except Exception as e:
    print(f"Could not load HF_TOKEN from Kaggle Secrets: {e}")

HF_TOKEN loaded securely from Kaggle Secrets.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/sample_submission.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/documents.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/agriculture-climate-slm-challenge/baseline_submission.csv


In [3]:
import os
import pandas as pd

# To find the file paths dynamically from /kaggle/input
train_path = None
test_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if filename == 'train_qa.csv':
            train_path = full_path
        elif filename == 'test_questions.csv':
            test_path = full_path

print(f"Train path: {train_path}")
print(f"Test path: {test_path}")

# Loading CSV files
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Displaying top of files
print("\n--- Train Head ---")
display(train_df.head())

print("\n--- Test Head ---")
display(test_df.head())

Train path: /kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
Test path: /kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv

--- Train Head ---


,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5



--- Test Head ---


,QuestionId,question,topic,crop,agro_zone
0,1001,How should I apply nitrogen to leaching-prone ...,crop_diseases,maize,sub_humid
1,1002,Grass gone in August — feed strategy?,livestock,livestock,semi_arid
2,1003,Red spots under my bean leaves during the rains.,crop_diseases,beans,highland
3,1004,Fresh cow dung on vegetable beds — safe?,fertiliser,general,sub_humid
4,1005,How much compost per hectare?,fertiliser,general,sub_humid


In [4]:
!pip install sentence-transformers -q

In [5]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load lightweight, high-performance dense embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r'\s+', ' ', text).strip()

# 2. Prepare rich question representations
train_clean = train_df.copy()
train_clean['clean_q'] = train_clean['question'].apply(clean_text)
train_clean['clean_a'] = train_clean['reference_answer'].apply(clean_text)

# Build search context combining metadata
def format_entry(row):
    crop = str(row.get('crop', '')).strip()
    zone = str(row.get('agro_zone', '')).strip()
    topic = str(row.get('topic', '')).strip()
    q = clean_text(row['question'])
    return f"{crop} {zone} {topic} {q}".strip()

train_texts = [format_entry(row) for _, row in train_clean.iterrows()]
test_texts = [format_entry(row) for _, row in test_df.iterrows()]

# 3. Compute Dense Semantic Embeddings
train_embeddings = embedder.encode(train_texts, convert_to_numpy=True, normalize_embeddings=True)
test_embeddings = embedder.encode(test_texts, convert_to_numpy=True, normalize_embeddings=True)

# 4. Dense Cosine Similarity
similarity_matrix = cosine_similarity(test_embeddings, train_embeddings)

# 5. Extract Best Matching Span / Sentence to minimize Levenshtein penalty
final_answers = []

for i in range(len(test_df)):
    test_row = test_df.iloc[i]
    t_crop = str(test_row.get('crop', '')).strip().lower()
    t_zone = str(test_row.get('agro_zone', '')).strip().lower()
    
    sim_scores = similarity_matrix[i].copy()
    
    # Apply soft domain bonus for matching crop and zone
    for j in range(len(train_clean)):
        train_row = train_clean.iloc[j]
        if t_crop and str(train_row.get('crop', '')).strip().lower() == t_crop:
            sim_scores[j] += 0.15
        if t_zone and str(train_row.get('agro_zone', '')).strip().lower() == t_zone:
            sim_scores[j] += 0.10
            
    best_idx = sim_scores.argmax()
    full_answer = train_clean.iloc[best_idx]['clean_a']
    
    # Split into individual sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', full_answer) if len(s.strip()) > 5]
    
    if len(sentences) > 1:
        # Score each candidate sentence against the test question
        sent_embeddings = embedder.encode(sentences, convert_to_numpy=True, normalize_embeddings=True)
        q_embedding = test_embeddings[i:i+1]
        sent_sims = cosine_similarity(q_embedding, sent_embeddings).flatten()
        best_sent = sentences[sent_sims.argmax()]
        
        # If the best sentence captures the core directive, use it
        final_answer = best_sent if len(best_sent) >= 20 else full_answer
    else:
        final_answer = full_answer
        
    final_answers.append(final_answer)

test_df['Answer'] = final_answers

# 6. Format and save
submission = test_df[['QuestionId', 'Answer']].copy()
submission['Answer'] = submission['Answer'].fillna("No answer provided.")
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Dense Semantic Retrieval with Sentence Trimming complete!")
display(submission.head(12))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense Semantic Retrieval with Sentence Trimming complete!


,QuestionId,Answer
0,1001,Only if rain or irrigation is imminent; otherw...
1,1002,"Harvest forage at boot stage, sun-dry on racks..."
2,1003,Reddish-brown pustules on leaf undersides in c...
3,1004,"Before hard seed set, leaving residues as surf..."
4,1005,Mucuna or lablab reduce erosion and suppress w...
5,1006,Stem borer tunneling kills the growing point i...
6,1007,Acidity binds phosphorus and limits nodulation...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,Use short-season drought-tolerant varieties al...


In [6]:
# Format submission
submission = test_df[['QuestionId', 'Answer']].copy()
submission['Answer'] = submission['Answer'].fillna("No answer provided.")

# Save directly to /kaggle/working/submission.csv
submission.to_csv("/kaggle/working/submission.csv", index=False)

# Verification checks
print(f"Saved: /kaggle/working/submission.csv")
print(f"Row count: {len(submission)} (expected: 12)")
print(f"Columns: {list(submission.columns)} (expected: ['QuestionId', 'Answer'])")
print(f"Null answers: {submission['Answer'].isna().sum()}")
print(f"Empty answers: {(submission['Answer'].str.strip() == '').sum()}")

display(submission)

Saved: /kaggle/working/submission.csv
Row count: 12 (expected: 12)
Columns: ['QuestionId', 'Answer'] (expected: ['QuestionId', 'Answer'])
Null answers: 0
Empty answers: 0


,QuestionId,Answer
0,1001,Only if rain or irrigation is imminent; otherw...
1,1002,"Harvest forage at boot stage, sun-dry on racks..."
2,1003,Reddish-brown pustules on leaf undersides in c...
3,1004,"Before hard seed set, leaving residues as surf..."
4,1005,Mucuna or lablab reduce erosion and suppress w...
5,1006,Stem borer tunneling kills the growing point i...
6,1007,Acidity binds phosphorus and limits nodulation...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,Use short-season drought-tolerant varieties al...
